# Stage 4 — Multi-Task Auxiliary Signal (6-Class GBDT)

In this stage, we implement multi-task learning using a Joint State-Space Multi-Class formulation. We define a 6-class joint target representing all combinations of the binary failure prediction target `failure_within_horizon` ($y \in \{0, 1\}$) and the train-only auxiliary target `failure_mode` ($m \in \{\text{none}, \text{bearing}, \text{battery}\}$).

By training a 6-class LightGBM classifier, we regularize the shared tree trunk to learn the specific signatures for different failure modes. During inference, we obtain the probability of failure within horizon by summing the probabilities of classes representing a failure (Classes 3, 4, and 5).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb
from src.data import load_split
from src.cv import get_folds, assert_disjoint_groups
from src.metrics import auprc, report_folds
from src.feature_panel import build_feature_panel
from src.modality_dropout import apply_modality_dropout

RANDOM_STATE = 42

## 1. Load Data Splits
We load train and test splits, and collect both target fields (`failure_within_horizon` and `failure_mode`) from the train split.

In [4]:
train = load_split("train")
tab, notes, signals, channel_names = train["tab"], train["notes"], train["signals"], train["channel_names"]
note_text = notes["maintenance_note"]
y = tab["failure_within_horizon"].values
m = tab["failure_mode"].astype(str).str.strip().values
print(f"Train split: {len(tab)} flights, {tab['drone_id'].nunique()} drones, base rate = {y.mean():.4f}")

test = load_split("test")
tab_test, notes_test, signals_test = test["tab"], test["notes"], test["signals"]
note_text_test = notes_test["maintenance_note"]
print(f"Test split: {len(tab_test)} flights, {tab_test['drone_id'].nunique()} drones")

Train split: 15872 flights, 620 drones, base rate = 0.1251
Test split: 7534 flights, 290 drones


## 2. Define Joint State-Space Targets
We map each train row to one of 6 classes corresponding to $(y, m)$ combinations:
* Class 0: $y=0, m=\text{none}$
* Class 1: $y=0, m=\text{battery}$
* Class 2: $y=0, m=\text{bearing}$
* Class 3: $y=1, m=\text{none}$
* Class 4: $y=1, m=\text{battery}$
* Class 5: $y=1, m=\text{bearing}$

In [5]:
joint_y = []
for y_val, m_val in zip(y, m):
    if y_val == 0:
        if m_val == "none":
            joint_y.append(0)
        elif m_val == "battery":
            joint_y.append(1)
        elif m_val == "bearing":
            joint_y.append(2)
        else:
            raise ValueError(f"Unknown m_val for y=0: {m_val}")
    elif y_val == 1:
        if m_val == "none":
            joint_y.append(3)
        elif m_val == "battery":
            joint_y.append(4)
        elif m_val == "bearing":
            joint_y.append(5)
        else:
            raise ValueError(f"Unknown m_val for y=1: {m_val}")
joint_y = np.array(joint_y)

unique, counts = np.unique(joint_y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {u} count: {c}")

Class 0 count: 13817
Class 1 count: 23
Class 2 count: 46
Class 3 count: 575
Class 4 count: 452
Class 5 count: 959


## 3. Build Clean Feature Panels
We build clean feature panels for both train and test splits.

In [6]:
panel_clean, cat_cols, text_cols = build_feature_panel(tab, note_text, signals, channel_names)
feature_cols = [c for c in panel_clean.columns if c != "flight_id"]
print("Train panel clean shape:", panel_clean.shape)

panel_test, _, _ = build_feature_panel(tab_test, note_text_test, signals_test, channel_names)
print("Test panel shape:", panel_test.shape)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Train panel clean shape: (15872, 1181)
Test panel shape: (7534, 1181)


## 4. Train Multiclass GBDT with Modality Dropout
We use the Stratified Group K-Fold scheme. On each fold, we apply synthetic modality dropout (15% rate) during training. We evaluate out-of-fold validation performance by summing the probabilities of classes 3, 4, and 5 to obtain $P(y=1)$.

In [7]:
folds = list(get_folds(tab, n_splits=5, seed=RANDOM_STATE))

lgb_params = dict(
    objective="multiclass",
    num_class=6,
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

oof_preds = np.zeros(len(tab))
test_preds_folds = []
fold_scores = []

for fold_i, (tr_idx, val_idx) in enumerate(folds):
    assert_disjoint_groups(tab, tr_idx, val_idx)
    
    # Apply modality dropout to train indices
    sig_aug, txt_aug, _, _ = apply_modality_dropout(
        signals, note_text, tr_idx, p_signal=0.15, p_text=0.15, seed=RANDOM_STATE + fold_i
    )
    
    # Build feature panel for augmented train split
    panel_aug, _, _ = build_feature_panel(tab, txt_aug, sig_aug, channel_names)
    
    # Train model
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        panel_aug.loc[tr_idx, feature_cols], 
        joint_y[tr_idx], 
        categorical_feature=cat_cols
    )
    
    # Predict probabilities on validation fold
    val_probs = model.predict_proba(panel_clean.loc[val_idx, feature_cols])
    # Marginalize target P(y=1) = p3 + p4 + p5
    val_pred = val_probs[:, 3] + val_probs[:, 4] + val_probs[:, 5]
    oof_preds[val_idx] = val_pred
    
    score = auprc(y[val_idx], val_pred)
    fold_scores.append(score)
    print(f"Fold {fold_i} Validation AUPRC: {score:.5f}")
    
    # Predict on test set
    test_probs = model.predict_proba(panel_test[feature_cols])
    test_pred = test_probs[:, 3] + test_probs[:, 4] + test_probs[:, 5]
    test_preds_folds.append(test_pred)

Fold 0 Validation AUPRC: 0.51136
Fold 1 Validation AUPRC: 0.50349
Fold 2 Validation AUPRC: 0.49945
Fold 3 Validation AUPRC: 0.53311
Fold 4 Validation AUPRC: 0.52470


## 5. Evaluation Summary
We evaluate the multi-task model performance and compare it against the Stage 3 baseline.

In [8]:
report_folds(fold_scores, "Multitask GBDT")
pooled_oof_score = auprc(y, oof_preds)
print(f"Pooled OOF AUPRC: {pooled_oof_score:.5f}")

Multitask GBDT: fold AUPRCs = [0.5114, 0.5035, 0.4994, 0.5331, 0.5247]
Multitask GBDT: mean=0.5144  std=0.0127  (floor~0.125)
Pooled OOF AUPRC: 0.51341


## 6. Generate and Verify final Test Predictions
We average the predictions from the 5 fold models and export them to `submission.csv`.

In [9]:
final_test_preds = np.mean(test_preds_folds, axis=0)

submission = pd.DataFrame({
    "flight_id": tab_test["flight_id"],
    "failure_within_horizon": final_test_preds
})

submission.to_csv("../submission.csv", index=False)
print("Saved submission to ../submission.csv")

# Verify submission.csv properties
import os
sub_file = "../submission.csv"
assert os.path.exists(sub_file), "submission.csv was not created!"
lines = open(sub_file).readlines()
print(f"Number of lines in submission.csv: {len(lines)}")
assert len(lines) == len(tab_test) + 1, "Line count mismatch!"
print(submission["failure_within_horizon"].describe())

Saved submission to ../submission.csv
Number of lines in submission.csv: 7535
count    7534.000000
mean        0.070831
std         0.177704
min         0.000240
25%         0.006763
50%         0.012183
75%         0.033848
max         0.999679
Name: failure_within_horizon, dtype: float64
